In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('dataset\\train.csv')

In [3]:
df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [4]:
df.isnull().sum()

Index               0
geohash             0
day                 0
timestamp           0
demand              0
RoadType          600
NumberofLanes       0
LargeVehicles       0
Landmarks           0
Temperature      2495
Weather           797
dtype: int64

In [5]:
df['RoadType'].value_counts()

RoadType
Residential    69230
Street          3909
Highway         3560
Name: count, dtype: int64

In [6]:
df['Weather'].value_counts()

Weather
Sunny    27717
Rainy    20824
Foggy    20243
Snowy     7718
Name: count, dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.groupby('Weather')['Temperature'].agg(['count', 'median', 'std'])

,count,median,std
Weather,,,
Foggy,19584,16.475623,1.434400
Rainy,20153,11.141699,1.969005
Snowy,7478,4.403676,2.995841
Sunny,26818,23.049061,4.031399


In [10]:

# 1. Check overlap (optional but informative)
both_missing = df['Weather'].isna() & df['Temperature'].isna()
print(f"Rows missing both Weather and Temperature: {both_missing.sum()}")

# 2. Fill Weather FIRST — use "Unknown" (don't assume it's Sunny)
df['Weather_missing'] = df['Weather'].isna().astype(int)
df['Weather'] = df['Weather'].fillna('Unknown')

# 3. Now impute temperature by weather group
#    Groups with no temperature data fall back to global median
global_temp_median = df['Temperature'].median()
weather_temp_medians = df.groupby('Weather')['Temperature'].median()

df['Temperature_missing'] = df['Temperature'].isna().astype(int)
df['Temperature'] = df.apply(
    lambda row: weather_temp_medians.get(row['Weather'], global_temp_median)
    if pd.isna(row['Temperature']) else row['Temperature'],
    axis=1
)

Rows missing both Weather and Temperature: 797


In [11]:

df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,Temperature_missing,Weather_missing
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,Unknown,1,1
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,0,0
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,0,0
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,11.141699,Rainy,0,0
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,0,0


In [12]:
df.isnull().sum()

Index                    0
geohash                  0
day                      0
timestamp                0
demand                   0
RoadType               600
NumberofLanes            0
LargeVehicles            0
Landmarks                0
Temperature            797
Weather                  0
Temperature_missing      0
Weather_missing          0
dtype: int64

In [13]:
df['RoadType'] = df['RoadType'].fillna(df['RoadType'].mode()[0])

In [14]:
df['RoadType'].value_counts()

RoadType
Residential    69830
Street          3909
Highway         3560
Name: count, dtype: int64

In [17]:

# 1. Flag everything
df['Weather_missing'] = df['Weather'].isna().astype(int)
df['Temperature_missing'] = df['Temperature'].isna().astype(int)
df['Both_missing'] = (df['Weather'].isna() & df['Temperature'].isna()).astype(int)

# 2. Fill weather first
df['Weather'] = df['Weather'].fillna('Unknown')

# 3. Calculate medians per weather group + global fallback
global_temp_median = df['Temperature'].median()
weather_medians = df.groupby('Weather')['Temperature'].median()

# 4. Fill temperature: use the weather median when it exists, otherwise global median
def fill_temp(row):
    if pd.isna(row['Temperature']):
        weather_median = weather_medians.get(row['Weather'])
        return global_temp_median if pd.isna(weather_median) else weather_median
    return row['Temperature']

df['Temperature'] = df.apply(fill_temp, axis=1)

In [18]:
df.isnull().sum()

Index                  0
geohash                0
day                    0
timestamp              0
demand                 0
RoadType               0
NumberofLanes          0
LargeVehicles          0
Landmarks              0
Temperature            0
Weather                0
Temperature_missing    0
Weather_missing        0
Both_missing           0
dtype: int64

In [26]:
df['geohash'].nunique()

1249

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Build model data from raw file to avoid accidental leakage from earlier notebook mutations
model_df = pd.read_csv('dataset\\train.csv')

# Optional index-like column
if 'Index' in model_df.columns:
    model_df = model_df.drop(columns=['Index'])

# --- Geohash -> latitude/longitude (pure Python decoder, no extra package needed) ---
_BASE32 = '0123456789bcdefghjkmnpqrstuvwxyz'
_BITS = [16, 8, 4, 2, 1]
_DECODE_MAP = {c: i for i, c in enumerate(_BASE32)}

def decode_geohash(gh):
    if pd.isna(gh):
        return np.nan, np.nan

    gh = str(gh).strip().lower()
    lat_range = [-90.0, 90.0]
    lon_range = [-180.0, 180.0]
    even = True

    for ch in gh:
        cd = _DECODE_MAP.get(ch)
        if cd is None:
            return np.nan, np.nan

        for mask in _BITS:
            if even:
                mid = (lon_range[0] + lon_range[1]) / 2
                if cd & mask:
                    lon_range[0] = mid
                else:
                    lon_range[1] = mid
            else:
                mid = (lat_range[0] + lat_range[1]) / 2
                if cd & mask:
                    lat_range[0] = mid
                else:
                    lat_range[1] = mid
            even = not even

    lat = (lat_range[0] + lat_range[1]) / 2
    lon = (lon_range[0] + lon_range[1]) / 2
    return lat, lon

decoded = model_df['geohash'].apply(decode_geohash)
model_df['geo_lat'] = decoded.str[0]
model_df['geo_lon'] = decoded.str[1]

# --- Timestamp features ---
time_parts = model_df['timestamp'].astype(str).str.split(':', expand=True)
model_df['hour'] = pd.to_numeric(time_parts[0], errors='coerce')
model_df['minute'] = pd.to_numeric(time_parts[1], errors='coerce').fillna(0)
model_df['minute_of_day'] = model_df['hour'] * 60 + model_df['minute']
model_df['minute_of_day_sin'] = np.sin(2 * np.pi * model_df['minute_of_day'] / 1440.0)
model_df['minute_of_day_cos'] = np.cos(2 * np.pi * model_df['minute_of_day'] / 1440.0)

# --- Robust missing-value handling ---
model_df['Weather'] = model_df['Weather'].fillna('Unknown')
model_df['RoadType'] = model_df['RoadType'].fillna(model_df['RoadType'].mode()[0])

temp_by_weather = model_df.groupby('Weather')['Temperature'].transform('median')
model_df['Temperature'] = model_df['Temperature'].fillna(temp_by_weather)
model_df['Temperature'] = model_df['Temperature'].fillna(model_df['Temperature'].median())

binary_map = {'Allowed': 1, 'Not Allowed': 0, 'Yes': 1, 'No': 0}
model_df['LargeVehicles'] = model_df['LargeVehicles'].map(binary_map)
model_df['Landmarks'] = model_df['Landmarks'].map(binary_map)

# --- Target and features ---
y = model_df['demand']
X = model_df.drop(columns=['demand', 'geohash', 'timestamp'])

categorical_cols = ['Weather', 'RoadType']
numeric_cols = [c for c in X.columns if c not in categorical_cols]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols),
])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_prepared = preprocessor.fit_transform(X_train)
X_val_prepared = preprocessor.transform(X_val)

print('Train matrix shape:', X_train_prepared.shape)
print('Validation matrix shape:', X_val_prepared.shape)

Train matrix shape: (61839, 20)
Validation matrix shape: (15460, 20)


In [28]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score

lgbm_model = LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=63,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

lgbm_model.fit(
    X_train_prepared,
    y_train,
    eval_set=[(X_val_prepared, y_val)],
    eval_metric='rmse',
)

lgbm_val_pred = lgbm_model.predict(X_val_prepared)
lgbm_rmse = np.sqrt(mean_squared_error(y_val, lgbm_val_pred))
lgbm_r2 = r2_score(y_val, lgbm_val_pred)

print(f'LightGBM Validation RMSE: {lgbm_rmse:.6f}')
print(f'LightGBM Validation R2: {lgbm_r2:.6f}')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007868 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 639
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 20
[LightGBM] [Info] Start training from score 0.093784


c:\Users\nanin\OneDrive\Desktop\FlipkartGrid_Hackathon\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


LightGBM Validation RMSE: 0.040640
LightGBM Validation R2: 0.918378


In [29]:
test_df = pd.read_csv('dataset\\test.csv')

In [30]:
# Use whichever variable name exists in the notebook (test_df or tets_df)
if 'test_df' in globals():
    pred_df = test_df.copy()
elif 'tets_df' in globals():
    pred_df = tets_df.copy()
else:
    raise NameError("Neither 'test_df' nor 'tets_df' is defined. Please load the test CSV first.")

# Keep submission index before dropping columns
submission_index = pred_df['Index'].copy() if 'Index' in pred_df.columns else pd.Series(np.arange(len(pred_df)))

# Geohash -> latitude/longitude (uses decode_geohash from training cell)
decoded_test = pred_df['geohash'].apply(decode_geohash)
pred_df['geo_lat'] = decoded_test.str[0]
pred_df['geo_lon'] = decoded_test.str[1]

# Timestamp features
time_parts_test = pred_df['timestamp'].astype(str).str.split(':', expand=True)
pred_df['hour'] = pd.to_numeric(time_parts_test[0], errors='coerce')
pred_df['minute'] = pd.to_numeric(time_parts_test[1], errors='coerce').fillna(0)
pred_df['minute_of_day'] = pred_df['hour'] * 60 + pred_df['minute']
pred_df['minute_of_day_sin'] = np.sin(2 * np.pi * pred_df['minute_of_day'] / 1440.0)
pred_df['minute_of_day_cos'] = np.cos(2 * np.pi * pred_df['minute_of_day'] / 1440.0)

# Missing value handling aligned with training
pred_df['Weather'] = pred_df['Weather'].fillna('Unknown')
pred_df['RoadType'] = pred_df['RoadType'].fillna(model_df['RoadType'].mode()[0])
pred_df['Temperature'] = pred_df['Temperature'].fillna(
    pred_df.groupby('Weather')['Temperature'].transform('median')
)
pred_df['Temperature'] = pred_df['Temperature'].fillna(model_df['Temperature'].median())

# Binary features
binary_map = {'Allowed': 1, 'Not Allowed': 0, 'Yes': 1, 'No': 0}
pred_df['LargeVehicles'] = pred_df['LargeVehicles'].map(binary_map)
pred_df['Landmarks'] = pred_df['Landmarks'].map(binary_map)

# Match training features
X_test = pred_df.drop(columns=[c for c in ['Index', 'geohash', 'timestamp'] if c in pred_df.columns])
X_test_prepared = preprocessor.transform(X_test)

# Predict with trained LightGBM
test_pred = lgbm_model.predict(X_test_prepared)

submission = pd.DataFrame({
    'Index': submission_index.astype(int),
    'demand': test_pred
})

submission.to_csv('submission_lightgbm.csv', index=False)
print('Saved: submission_lightgbm.csv')
print(submission.head())

c:\Users\nanin\OneDrive\Desktop\FlipkartGrid_Hackathon\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Saved: submission_lightgbm.csv
   Index    demand
0      0  0.044979
1      1  0.048059
2      2  0.025334
3      3  0.037359
4      4  0.050883
